In [1]:
import time
import logging
from selenium import webdriver
from selenium.webdriver.common import by
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup



In [2]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# --- Selenium Setup ---
opts = Options()
opts.headless = False  # set True if you don't want browser window
driver = webdriver.Firefox(options=opts)



In [4]:
# while True:
#     try:
        # find the last button (usually "more")
more_button = driver.find_elements(by.By.TAG_NAME, "button")[-1]
driver.execute_script("arguments[0].scrollIntoView();", more_button)  # scroll to button
time.sleep(1)

# click it
more_button.click()
time.sleep(2)  # wait for new content to load

# re-parse the updated page
soup = BeautifulSoup(driver.page_source, 'html.parser')
a_tags = soup.select("a[href^='/detail']")

# extract info from the new soup
for detail in a_tags:
    span_tags = detail.find_all("span")
    price = None
    place = None
    room = None
    metter = None
    condition = None

    for i, span in enumerate(span_tags):
        text = span.get_text(strip=True)

        if price is None and "قیمت" in text and i + 1 < len(span_tags):
            price = span_tags[i + 1].get_text(strip=True)            
            place = span_tags[i +2 ].get_text(strip=True)


        if room is None and ("اتاق" in text or "خواب" in text) and i - 1 >= 0:
            room = span_tags[i - 1].get_text(strip=True)

        if metter is None and "متر" in text and i - 1 >= 0:
            metter = span_tags[i - 1].get_text(strip=True)

        if condition is None:
            if "ساله" in text and i - 1 >= 0:
                condition = f"{span_tags[i - 1].get_text(strip=True)} ساله"
            elif "نوساز" in text:
                condition = "نوساز"
            elif "کلید نخورده" in text:
                condition = "کلید نخورده"
        if price is None :
            continue
    print(f"قیمت: {price}, اتاق: {room}, متر: {metter}, وضعیت: {condition}  {place} ")
    print("*"* 40)

    # except Exception as e:
    #     print("No more button found or end of listings.")
    #     break


IndexError: list index out of range